# Ropedia Academy — B9 · Paper deep-dive & reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B9.ipynb)

> **A complete miniature NeRF training run, visualized by the photometric-loss curve and a target-vs-rendered pixel strip (with a floater-suppressing sparsity term).**
>
> 完整的微型 NeRF 训练，用光度损失曲线和「目标 vs 渲染」像素条可视化（含抑制漂浮物的稀疏项）。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B9

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt

# Minimal end-to-end NeRF: representation -> renderer -> photometric loss.
def posenc(x, L=4):
    return torch.cat([x] + [f(x*(2.0**i)) for i in range(L) for f in (torch.sin, torch.cos)], -1)
mlp = nn.Sequential(nn.Linear(3*(1+2*4), 128), nn.ReLU(), nn.Linear(128, 4))
def render(p, delta):
    o = mlp(posenc(p)); c, s = torch.sigmoid(o[..., :3]), torch.relu(o[..., 3])
    a = 1 - torch.exp(-s * delta)
    Tr = torch.cumprod(torch.cat([torch.ones(p.shape[0],1), 1-a+1e-10], 1)[:, :-1], 1)
    return (Tr*a).unsqueeze(-1).mul(c).sum(1), a

R, S = 16, 32                                      # toy "posed rays" dataset
t = torch.linspace(2,6,S); delta = (t[1]-t[0]).expand(R,S)
pts = torch.randn(R,1,3)*0.1 + torch.stack([torch.zeros(S),torch.zeros(S),t], -1)
gt  = torch.rand(R, 3)
opt = torch.optim.Adam(mlp.parameters(), 5e-3); hist = []
for _ in range(300):
    opt.zero_grad(); pix, a = render(pts, delta)
    photo = ((pix - gt)**2).mean()
    (photo + 1e-3*a.mean()).backward(); opt.step(); hist.append(photo.item())
print("final photometric loss:", round(hist[-1], 4))

fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.2))
ax[0].plot(hist); ax[0].set_yscale('log'); ax[0].set_title("photometric loss"); ax[0].set_xlabel("iter")
pix, _ = render(pts, delta)
ax[1].imshow(torch.stack([gt, pix.detach().clamp(0,1)]).reshape(2, R, 3))
ax[1].set_yticks([0,1]); ax[1].set_yticklabels(['target','rendered']); ax[1].set_title("pixels")
plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks